In [ ]:
# Setup: connect Google Drive and open the project folder (works in Google Colab)
import os, sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/NovelMate_ML_Project"
except ImportError:                      # running on your own computer instead of Colab
    PROJECT_DIR = "."
if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError("Upload the NovelMate_ML_Project folder to My Drive first (see README.md).")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.getcwd())

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from ml_utils import *

In [ ]:
DATA_DIR.mkdir(exist_ok=True); MODEL_DIR.mkdir(exist_ok=True)
if not RAW_FILE.exists():
    try:
        from google.colab import files
        print("Please upload books.csv ...")
        uploaded = files.upload()
        RAW_FILE.write_bytes(next(iter(uploaded.values())))
    except (ImportError, StopIteration):
        raise FileNotFoundError("Put books.csv into the data folder and run this cell again.")

books = pd.read_csv(RAW_FILE)
print("Shape:", books.shape)
books.head()

In [ ]:
missing = [c for c in FEATURES + [TARGET] if c not in books.columns]
if missing:
    raise ValueError(f"books.csv is missing required columns: {missing}")
print("Missing values in the columns we use:")
print(books[FEATURES + [TARGET]].isna().sum().to_string())
display(books[[YEAR] + COUNTS + [TARGET]].describe().round(2))

In [ ]:
data = books[FEATURES + [TARGET]].copy()
for col in [YEAR] + COUNTS + [TARGET]:
    data[col] = pd.to_numeric(data[col], errors="coerce")
data = data.dropna(subset=[TARGET])
data = data[data[TARGET].between(0, 5)].copy()
data[COUNTS] = data[COUNTS].where(data[COUNTS] >= 0)
data[LANG] = data[LANG].fillna("unknown").astype(str)
print("Rows after cleaning:", len(data))

In [ ]:
q1, q2 = data[TARGET].quantile([1 / 3, 2 / 3])
data[GROUP] = pd.cut(data[TARGET], [-np.inf, q1, q2, np.inf], labels=CLASS_LABELS, right=False).astype(str)
CLASS_RANGES = {"Low": f"below {q1:.2f}", "Medium": f"{q1:.2f} to {q2:.2f}", "High": f"{q2:.2f} and above"}
print(CLASS_RANGES)
data[GROUP].value_counts().reindex(CLASS_LABELS)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
data[TARGET].hist(bins=40, ax=ax[0], color="#8b3a2f")
for edge in (q1, q2): ax[0].axvline(edge, color="black", linestyle="--")
ax[0].set_title("Average rating (dashed = group borders)")
data[GROUP].value_counts().reindex(CLASS_LABELS).plot.bar(ax=ax[1], color="#4a5a6a", rot=0)
ax[1].set_title("Rating groups")
ax[2].scatter(np.log1p(data["ratings_count"]), data[TARGET], s=4, alpha=0.3, color="#8b3a2f")
ax[2].set_xlabel("log(1 + ratings_count)"); ax[2].set_ylabel(TARGET); ax[2].set_title("Ratings count vs rating")
plt.tight_layout(); plt.show()
display(data[[YEAR] + COUNTS + [TARGET]].corr().round(2))

In [ ]:
data.to_csv(CLEAN_FILE, index=False)

info = {
    "features": FEATURES,
    "languages": data[LANG].value_counts().head(8).index.tolist(),    # choices in the web form
    "class_labels": CLASS_LABELS, "class_ranges": CLASS_RANGES,
    "models": {"regression": {}, "classification": {}},
    "metrics": {"regression": [], "classification": []},
    "dataset_rows": int(len(data)),
}
joblib.dump(info, INFO_FILE)
print("Saved", CLEAN_FILE, "and", INFO_FILE)